In [ ]:
import re
import math
from collections import Counter, defaultdict
import nltk
from nltk.util import ngrams
from nltk.stem import PorterStemmer

nltk.download('punkt')

############################################################
# 1. Load and preprocess corpus
############################################################

def load_corpus(file_path):
    with open("/content/cat7.txt", "r", encoding="utf8") as f:
        text = f.read().lower()

    words = re.findall(r'\w+', text)
    return words


############################################################
# 2. Build frequency dictionary
############################################################

def build_frequency(words):
    return Counter(words)


############################################################
# 3. Build N-gram model (Bigram)
############################################################

def build_ngram_model(words, n=2):
    model = defaultdict(int)

    for gram in ngrams(words, n):
        model[gram] += 1

    return model


def ngram_probability(model, word):
    total = sum(model.values())
    return model.get(word, 0) / total if total > 0 else 0


############################################################
# 4. Morphological similarity using stemming
############################################################

stemmer = PorterStemmer()

def morphological_similarity(word1, word2):

    stem1 = stemmer.stem(word1)
    stem2 = stemmer.stem(word2)

    if stem1 == stem2:
        return 1.0

    if stem1 in word2 or stem2 in word1:
        return 0.7

    return 0.3


############################################################
# 5. Edit distance (Levenshtein)
############################################################

def edit_distance(a, b):

    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]

    for i in range(len(a)+1):
        dp[i][0] = i

    for j in range(len(b)+1):
        dp[0][j] = j

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):

            cost = 0 if a[i-1] == b[j-1] else 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[-1][-1]


############################################################
# 6. Candidate generation
############################################################

def generate_candidates(word, vocabulary, max_distance=2):

    candidates = []

    for v in vocabulary:
        if abs(len(v) - len(word)) <= max_distance:
            if edit_distance(word, v) <= max_distance:
                candidates.append(v)

    return candidates


############################################################
# 7. Candidate scoring
############################################################

def score_candidate(word, candidate, freq_dict):

    total_words = sum(freq_dict.values())

    # language probability
    pw = freq_dict[candidate] / total_words

    # error probability
    distance = edit_distance(word, candidate)
    pxw = 1 / (distance + 1)

    # morphological similarity
    morph = morphological_similarity(word, candidate)

    score = pw * pxw * morph

    return score


############################################################
# 8. Spell correction
############################################################

def correct_spelling(word, freq_dict):

    vocabulary = set(freq_dict.keys())

    candidates = generate_candidates(word, vocabulary)

    if not candidates:
        return word

    best_candidate = max(
        candidates,
        key=lambda c: score_candidate(word, c, freq_dict)
    )

    return best_candidate


############################################################
# 9. Train model on corpus
############################################################

def train_model(corpus_path):

    words = load_corpus(corpus_path)

    freq_dict = build_frequency(words)

    ngram_model = build_ngram_model(words)

    return freq_dict, ngram_model


############################################################
# 10. Example usage
############################################################

if __name__ == "__main__":

    corpus_file = "large_corpus.txt"

    freq_dict, ngram_model = train_model(corpus_file)

    while True:
        word = input("Enter word: ")

        corrected = correct_spelling(word, freq_dict)

        print("Corrected word:", corrected)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Enter word: improve
Corrected word: improve
Enter word: natral
Corrected word: natural
Enter word: intgence
Corrected word: intgence
Enter word: intelgence
Corrected word: intelligence
Enter word: ca
Corrected word: can
Enter word: wld
Corrected word: world
Enter word: programming
Corrected word: programming
Enter word: procing
Corrected word: procing
Enter word: proceing
Corrected word: processing
